# 🦀 C2Rust RL — Training with TRL GRPO + OpenEnv

This notebook trains **Qwen2.5-Coder-1.5B-Instruct** to migrate C code to safe Rust using:
- **OpenEnv** C2Rust environment (running on HF Space) as the reward oracle
- **TRL GRPOTrainer** for the RL fine-tuning loop
- **Rust compiler** (`cargo check`) as the reward signal — no human labels

Designed to run on a **free Colab T4 GPU** (~15 min for a mini training run).

## What happens
1. We install Rust + dependencies
2. We clone the C2Rust repo and connect to the OpenEnv reward oracle
3. We load Qwen2.5-Coder-1.5B at 4-bit with QLoRA adapters
4. TRL GRPOTrainer runs online RL — generating Rust, scoring with the compiler, updating the model
5. We plot the reward and loss curves

## Step 1 — Install System Dependencies

We need the Rust toolchain (`cargo check` is the reward oracle) and `libclang` for static analysis.

In [ ]:
# Install Rust (stable) + libclang
!apt-get update -qq && apt-get install -y -qq libclang-dev
!curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --default-toolchain stable --profile minimal
import os
os.environ['PATH'] = '/root/.cargo/bin:' + os.environ['PATH']
!cargo --version

## Step 2 — Install Python Dependencies

In [ ]:
# Core training stack
!pip install -q \
    "torch>=2.0.0" \
    "transformers>=4.40.0" \
    "peft>=0.10.0" \
    "trl>=0.12.0" \
    "datasets>=2.18.0" \
    "bitsandbytes>=0.41.0" \
    "accelerate>=0.28.0"

# OpenEnv client (to connect to the HF Space reward server)
!pip install -q openenv-core fastmcp

# Utilities
!pip install -q libclang matplotlib pandas wandb

## Step 3 — Clone the Repo + Set Up Reward Function

In [ ]:
import os, sys

# Clone the C2Rust repo (or mount from Drive if you already have it)
if not os.path.exists('/content/shiftenv-v2'):
    !git clone https://github.com/sumit-s-nair/shiftenv-v2 /content/shiftenv-v2

sys.path.insert(0, '/content/shiftenv-v2')
os.chdir('/content/shiftenv-v2')

# Verify reward function loads
from reward import compute_reward
score, info = compute_reward("fn main() { println!(\"Hello\"); }")
print(f"Reward function OK — test score: {score:.3f}")

## Step 4 — Connect to the OpenEnv HF Space

The OpenEnv environment server is running on HF Space. We use it for:
- Getting C source files to translate (`reset()`)
- Scoring Rust code without needing to call the reward function locally

> **Note:** The local `compute_reward()` from `reward.py` does the same thing as the remote tool — both call `cargo check`. We use **local** reward in the TRL training loop (faster, no network latency), and the HF Space env for interactive exploration.

In [ ]:
from env.client import C2RustEnv

# ⬇️  Replace with your actual HF Space URL once deployed
HF_SPACE_URL = "https://YOUR-USERNAME-c2rust-rl.hf.space"

try:
    with C2RustEnv(base_url=HF_SPACE_URL) as env:
        obs = env.reset()
        c_source = obs.observation.metadata.get("c_source", "")
        module = obs.observation.metadata.get("module_name", "")
        print(f"✅ Connected to HF Space env")
        print(f"   Module to translate: {module}")
        print(f"   C source ({len(c_source)} chars): {c_source[:200]}...")

        # Score a trivial Rust stub
        result = env.call_tool("score_rust_code",
                               rust_code="fn placeholder() {}",
                               module_name=module)
        print(f"   Reward for stub: {result}")
except Exception as e:
    print(f"⚠️  Could not connect to HF Space ({e})")
    print("   Falling back to local reward oracle — training will still work!")

## Step 5 — Prepare the Training Dataset

Each training example is a prompt asking the model to translate a C file. The reward function scores the model's Rust output using `cargo check`.

In [ ]:
import glob
from datasets import Dataset

SYSTEM_PROMPT = (
    "You are a compiler-like C-to-Rust translator. "
    "Output ONLY valid Rust source code. "
    "No explanations, no markdown fences, no commentary."
)

def build_prompt(c_source: str) -> str:
    return (
        f"Convert the following C code to safe, idiomatic Rust.\n\n"
        f"```c\n{c_source}\n```"
    )

# Collect all .c files from the test suite
c_files = sorted(glob.glob('tests/**/*.c', recursive=True))
print(f"Found {len(c_files)} C files")

# For Colab demo: use easy + data_basics only (fits in ~15 min)
demo_files = [f for f in c_files if any(d in f for d in ['data_basics', 'easy'])]
print(f"Using {len(demo_files)} files for demo run")

records = []
for path in demo_files:
    with open(path, encoding='utf-8', errors='ignore') as f:
        src = f.read()
    if len(src.strip()) < 10:
        continue
    records.append({
        "prompt": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": build_prompt(src)},
        ],
        "module_name": os.path.splitext(os.path.basename(path))[0],
    })

dataset = Dataset.from_list(records)
print(f"Dataset: {len(dataset)} examples")
print(dataset[0]["prompt"][1]["content"][:300])

## Step 6 — Define the Reward Function for TRL

TRL's `GRPOTrainer` expects a reward function with the signature:
```python
def reward_fn(prompts, completions, **kwargs) -> list[float]
```

In [ ]:
import re
from reward import compute_reward

def clean_rust_output(text: str) -> str:
    """Strip markdown fences and leading prose from model output."""
    text = re.sub(r'```[a-zA-Z]*', '', text).replace('```', '')
    lines = text.splitlines()
    for i, line in enumerate(lines):
        if line.strip().startswith(('pub ', 'fn ', 'use ', 'mod ', '#[', 'struct ', 'enum ')):
            return '\n'.join(lines[i:]).strip()
    return text.strip()

def c2rust_reward(prompts, completions, module_name=None, **kwargs):
    """
    Reward function for TRL GRPOTrainer.
    Scores each completion using the Rust compiler.
    """
    rewards = []
    for i, completion in enumerate(completions):
        # completions is a list of lists (one inner list per prompt)
        if isinstance(completion, list):
            text = completion[0].get('content', '') if completion else ''
        else:
            text = str(completion)

        rust_code = clean_rust_output(text)
        mod_name = (module_name[i // 1] if module_name else 'module')  # 1 completion per prompt

        try:
            score, info = compute_reward(rust_code, module_name=mod_name, timeout=20)
        except Exception:
            score = 0.0

        rewards.append(score)
    return rewards

# Quick sanity check
test_rewards = c2rust_reward(
    prompts=["test"],
    completions=[[{"role": "assistant", "content": "fn main() { println!(\"hi\"); }"}]],
    module_name=["test"],
)
print(f"Reward function OK — test reward: {test_rewards[0]:.3f}")

## Step 7 — Load Model (Qwen2.5-Coder-1.5B, 4-bit QLoRA)

We use `Qwen2.5-Coder-1.5B-Instruct` — fits comfortably on a T4 (16 GB) with room for activations and LoRA gradients.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

MODEL_NAME = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
# For a stronger run (needs ~6 GB VRAM), swap to:
# MODEL_NAME = "Qwen/Qwen2.5-Coder-3B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print(f"Loading {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

base_model = prepare_model_for_kbit_training(base_model, use_gradient_checkpointing=True)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()
print("✅ Model loaded")

## Step 8 — Configure and Run TRL GRPOTrainer

In [ ]:
from trl import GRPOTrainer, GRPOConfig

grpo_config = GRPOConfig(
    output_dir="/content/c2rust_grpo_output",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=5e-6,
    num_generations=4,           # GRPO group size — 4 candidates per prompt
    max_new_tokens=512,
    temperature=1.2,             # Higher temp = more diverse rollouts = better signal
    top_p=0.95,
    logging_steps=1,
    save_steps=10,
    report_to="none",            # Set to "wandb" if you have W&B configured
    bf16=True,
    remove_unused_columns=False,
)

trainer = GRPOTrainer(
    model=model,
    reward_funcs=c2rust_reward,
    args=grpo_config,
    train_dataset=dataset,
    processing_class=tokenizer,
)

print("Starting GRPO training...")
print(f"  Model: {MODEL_NAME}")
print(f"  Dataset: {len(dataset)} C files")
print(f"  Group size: 4 (4 Rust candidates generated per file)")
print(f"  Reward: cargo check + unsafe/warning/idiom scoring")
print()

train_result = trainer.train()

## Step 9 — Plot Training Curves

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

log_history = trainer.state.log_history
df = pd.DataFrame(log_history)

# Extract relevant columns (TRL GRPO logs these keys)
loss_cols = [c for c in df.columns if 'loss' in c.lower()]
reward_cols = [c for c in df.columns if 'reward' in c.lower()]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("C2Rust RL Training — Qwen2.5-Coder-1.5B GRPO", fontsize=14)

if loss_cols:
    df_loss = df[['step'] + loss_cols].dropna()
    for col in loss_cols:
        axes[0].plot(df_loss['step'], df_loss[col], label=col, marker='o', markersize=3)
    axes[0].set_title("Policy Loss")
    axes[0].set_xlabel("Step")
    axes[0].set_ylabel("Loss")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

if reward_cols:
    df_rew = df[['step'] + reward_cols].dropna()
    for col in reward_cols:
        axes[1].plot(df_rew['step'], df_rew[col], label=col, marker='o', markersize=3)
    axes[1].axhline(1.0, color='green', linestyle='--', alpha=0.5, label='Perfect (1.0)')
    axes[1].set_title("Compiler Reward")
    axes[1].set_xlabel("Step")
    axes[1].set_ylabel("Reward [0–1]")
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("/content/training_curves.png", dpi=150)
plt.show()
print("Saved to /content/training_curves.png")

## Step 10 — Test the Trained Model

Let's see how the fine-tuned model translates a C file and what reward it gets.

In [ ]:
import json

# Pick a test C file
test_c = records[0]
c_source = test_c["prompt"][1]["content"]
module_name = test_c["module_name"]

print(f"Testing on module: {module_name}")
print("=" * 50)

# Tokenize the prompt using the chat template
messages = test_c["prompt"]
prompt_text = tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)
inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)

# Generate Rust translation
model.eval()
with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=512,
        do_sample=True,
        temperature=0.7,
        pad_token_id=tokenizer.eos_token_id,
    )

generated = tokenizer.decode(output_ids[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
rust_code = clean_rust_output(generated)

print("Generated Rust:")
print(rust_code[:1000])
print()

# Score it
reward, info = compute_reward(rust_code, module_name=module_name)
print(f"Reward:          {reward:.4f}")
print(f"Compiled:        {info.error_count == 0}")
print(f"Unsafe blocks:   {info.unsafe_count}")
print(f"Warnings:        {info.warning_count}")
print(f"Errors:          {info.error_count}")
if info.errors:
    print("Compiler errors:")
    for e in info.errors[:5]:
        print(f"  {e}")

## Step 11 — Save the LoRA Adapters

Save the trained adapters so you can push them to the Hub or load them later.

In [ ]:
ADAPTER_SAVE_PATH = "/content/c2rust_lora_adapters"
model.save_pretrained(ADAPTER_SAVE_PATH)
tokenizer.save_pretrained(ADAPTER_SAVE_PATH)
print(f"✅ Adapters saved to {ADAPTER_SAVE_PATH}")

# Optional: push to Hub
# from huggingface_hub import login
# login(token="hf_...")
# model.push_to_hub("YOUR-USERNAME/c2rust-qwen-1.5b-lora")
# tokenizer.push_to_hub("YOUR-USERNAME/c2rust-qwen-1.5b-lora")

---

## Summary

In this notebook we:
1. Set up the Rust compiler as the reward oracle (no human labels needed)
2. Loaded Qwen2.5-Coder-1.5B-Instruct at 4-bit with QLoRA
3. Used TRL's GRPOTrainer for online RL — the model trained on its own Rust generations
4. Showed that the compiler reward signal improves translation quality over training

**Links:**
- 🤗 HF Space (live env): [YOUR-USERNAME/c2rust-rl](https://huggingface.co/spaces/YOUR-USERNAME/c2rust-rl)
- 📝 Blog post: [blog.md](https://huggingface.co/spaces/YOUR-USERNAME/c2rust-rl/blob/main/blog.md)
- 🦀 GitHub: [shiftenv-v2](https://github.com/sumit-s-nair/shiftenv-v2)

The full 7B model training run (on A10G with the complete medium/hard test suite) is
documented in the W&B run linked from the README.